# 종합 실습 (종합 프로젝트) — 새 오염 데이터셋, 처음부터 끝까지

## [환경 준비]

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'outputs/' 가 만들어집니다)
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.0.1
pandas: 3.0.6
저장 폴더: outputs


In [2]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


## 시나리오 1 — 품질 진단 (Part 2 적용)

In [3]:
# 시나리오 1 - 품질 진단

def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base

qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
store_id,str,0,0.00,7,NaN,False
date_str,str,0,0.00,9,NaN,False
item,str,0,0.00,6,NaN,False
quantity,int64,0,0.00,5,14.56,False
unit_price,int64,0,0.00,5,0.00,False
revenue,float64,60,2.77,26,17.30,False


## 시나리오 2 — 정제 파이프라인 (Part 3·6 적용)

In [4]:
# 심화 1. full_pipeline 모듈화
from pipeline import full_pipeline, b_dedup, b_strip_store, b_unify_item, b_parse_date
bakery_clean, refunds_bakery = full_pipeline(bakery)

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")

정제 전: (2164, 6)  →  정제 후: (1784, 6)
환불 보관: 1건
item 종류: <ArrowStringArray>
['쿠키', '케이크', '샌드위치', '크루아상', '식빵']
Length: 5, dtype: str


## 시나리오 3 — 변환 + Parquet 저장 (Part 4·5 적용)

In [5]:
# 시나리오 3 — KPI 표 작성

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR / 'bakery_store_month.parquet'}, {OUT_DIR / 'bakery_item_kpi.parquet'}")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6834000.0,7243500.0,5884000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S04,6471500.0,7105000.0,6744500.0
S05,8681500.0,4102000.0,4960000.0
S06,9156000.0,5095500.0,6796500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: outputs/bakery_store_month.parquet, outputs/bakery_item_kpi.parquet


In [6]:
# 시나리오 3-2 — bakery_clean을 Parquet / CSV로 각각 저장하고 용량 비교
clean_pq = OUT_DIR / "bakery_clean.parquet"
clean_csv = OUT_DIR / "bakery_clean.csv"

bakery_clean.to_parquet(clean_pq, index=False)
bakery_clean.to_csv(clean_csv, index=False)

pq_kb = clean_pq.stat().st_size / 1024
csv_kb = clean_csv.stat().st_size / 1024

print(f"CSV     : {csv_kb:.1f} KB")
print(f"Parquet : {pq_kb:.1f} KB")
print(f"→ Parquet은 CSV의 {pq_kb / csv_kb * 100:.1f}% ({csv_kb - pq_kb:.1f} KB 차이)")

# 자료형 보존 확인: Parquet은 date가 datetime으로, CSV는 문자열로 돌아옴
print("\n[다시 읽었을 때 date 컬럼 dtype]")
print("Parquet:", pd.read_parquet(clean_pq)["date"].dtype)
print("CSV    :", pd.read_csv(clean_csv)["date"].dtype)

CSV     : 83.9 KB
Parquet : 9.0 KB
→ Parquet은 CSV의 10.7% (74.9 KB 차이)

[다시 읽었을 때 date 컬럼 dtype]
Parquet: datetime64[us]
CSV    : str


## 결정 로그 표 

In [7]:
s1 = b_dedup(bakery)
n_dup = len(bakery) - len(s1)                                   # 중복 제거 건수

n_space = (s1["store_id"] != s1["store_id"].str.strip()).sum()  # 공백 있던 store_id 수
s2 = b_strip_store(s1)

n_cake = (s2["item"] == "케익").sum()                           # '케익' 표기 수
s3 = b_unify_item(s2)

s4 = b_parse_date(s3)
n_nat = s4["date"].isna().sum()                                 # 날짜 파싱 실패 수

n_refund = (s4["revenue"] < 0).sum()                            # 환불(음수) 수
n_missing = s4["revenue"].isna().sum()                          # revenue 결측 수

In [8]:
decisions = []

def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})

log_decision(
    "1. 중복 제거",
    "drop_duplicates()로 완전 중복 행 제거",
    "모든 컬럼 값이 같은 행은 같은 거래의 중복 입력으로 간주 (값 종류가 적어 우연히 겹친 행일 가능성은 한계로 남김)",
    f"{n_dup}건 제거",
)
log_decision(
    "2. store_id 정리",
    "str.strip()으로 앞뒤 공백 제거",
    "' S01 '과 'S01'은 같은 매장인데 groupby에서 따로 집계됨",
    f"{n_space}건 정리 (매장 7→6종)",
)
log_decision(
    "3. 상품명 통일",
    "'케익' → '케이크' 치환",
    "같은 상품의 표기 차이로 상품별 KPI가 둘로 쪼개짐",
    f"{n_cake}건 변환 (상품 6→5종)",
)
log_decision(
    "4. 날짜 파싱",
    "pd.to_datetime(format='mixed')로 변환 후 date_str 삭제",
    "'-', '/', 구분자 없음 3가지 포맷이 섞여 월별 집계 불가",
    f"파싱 실패 {n_nat}건",
)
log_decision(
    "5. 환불 분리",
    "revenue < 0 행을 refunds_bakery에 보관 후 본 분석에서 제외",
    "음수가 섞이면 매출 KPI가 왜곡됨, 삭제 대신 보관해 원인 분석에 활용",
    f"{n_refund}건 분리",
)
log_decision(
    "6. 결측 제거",
    "revenue 결측 행 drop",
    "결측률 약 3%로 낮고, 평균 등으로 채우면 없는 매출이 생겨 KPI 왜곡",
    f"{n_missing}건 제거 (원본 60건 중 2건은 중복 제거 단계에서 이미 제거)",
)

# 표 DataFrame 생성 및 출력
decision_log = pd.DataFrame(decisions)
# display(decision_log)

md_table = "| 단계 | 액션 | 근거 | 결과 |\n|---|---|---|---|\n"
for d in decisions:
    md_table += f"| {d['step']} | {d['action']} | {d['reason']} | {d['result']} |\n"
print(md_table)

| 단계 | 액션 | 근거 | 결과 |
|---|---|---|---|
| 1. 중복 제거 | drop_duplicates()로 완전 중복 행 제거 | 모든 컬럼 값이 같은 행은 같은 거래의 중복 입력으로 간주 (값 종류가 적어 우연히 겹친 행일 가능성은 한계로 남김) | 321건 제거 |
| 2. store_id 정리 | str.strip()으로 앞뒤 공백 제거 | ' S01 '과 'S01'은 같은 매장인데 groupby에서 따로 집계됨 | 10건 정리 (매장 7→6종) |
| 3. 상품명 통일 | '케익' → '케이크' 치환 | 같은 상품의 표기 차이로 상품별 KPI가 둘로 쪼개짐 | 8건 변환 (상품 6→5종) |
| 4. 날짜 파싱 | pd.to_datetime(format='mixed')로 변환 후 date_str 삭제 | '-', '/', 구분자 없음 3가지 포맷이 섞여 월별 집계 불가 | 파싱 실패 0건 |
| 5. 환불 분리 | revenue < 0 행을 refunds_bakery에 보관 후 본 분석에서 제외 | 음수가 섞이면 매출 KPI가 왜곡됨, 삭제 대신 보관해 원인 분석에 활용 | 1건 분리 |
| 6. 결측 제거 | revenue 결측 행 drop | 결측률 약 3%로 낮고, 평균 등으로 채우면 없는 매출이 생겨 KPI 왜곡 | 58건 제거 (원본 60건 중 2건은 중복 제거 단계에서 이미 제거) |



| 단계 | 액션 | 근거 | 결과 |
|---|---|---|---|
| 1. 중복 제거 | drop_duplicates()로 완전 중복 행 제거 | 모든 컬럼 값이 같은 행은 같은 거래의 중복 입력으로 간주 (값 종류가 적어 우연히 겹친 행일 가능성은 한계로 남김) | 321건 제거 |
| 2. store_id 정리 | str.strip()으로 앞뒤 공백 제거 | ' S01 '과 'S01'은 같은 매장인데 groupby에서 따로 집계됨 | 10건 정리 (매장 7→6종) |
| 3. 상품명 통일 | '케익' → '케이크' 치환 | 같은 상품의 표기 차이로 상품별 KPI가 둘로 쪼개짐 | 8건 변환 (상품 6→5종) |
| 4. 날짜 파싱 | pd.to_datetime(format='mixed')로 변환 후 date_str 삭제 | '-', '/', 구분자 없음 3가지 포맷이 섞여 월별 집계 불가 | 파싱 실패 0건 |
| 5. 환불 분리 | revenue < 0 행을 refunds_bakery에 보관 후 본 분석에서 제외 | 음수가 섞이면 매출 KPI가 왜곡됨, 삭제 대신 보관해 원인 분석에 활용 | 1건 분리 |
| 6. 결측 제거 | revenue 결측 행 drop | 결측률 약 3%로 낮고, 평균 등으로 채우면 없는 매출이 생겨 KPI 왜곡 | 58건 제거 (원본 60건 중 2건은 중복 제거 단계에서 이미 제거) |


# 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그
- 기간: 2025-04-01 ~ 2025-06-30 (3개월) 
- 원본 행 수 / 정제 후 행 수: 2,164행 → 1,784행

## 2. 발견한 품질 문제
- 결측: revenue 60건 (2.77%), 그 외 컬럼은 결측 없음
- 중복: 완전 중복 321건 (의도적으로 넣은 중복은 4건, 나머지는 값 종류가 적어 우연히 겹친 행)
- 이상치: revenue 음수(-45,000) 1건, quantity=50 의심값 (IQR 이상치 비율 quantity 14.56%, revenue 17.30%)
- 표기 혼재: store_id 공백(' S01 ') 10건, item '케익'/'케이크' 8건, date_str 포맷 3종(-, /, 구분자 없음)

## 3. 처리 결정과 근거 (5줄로)
1. 중복 행 321건 → 제거 (모든 값이 같은 행은 중복 입력으로 간주)
2. store_id 공백 → strip()으로 정리 (같은 매장이 따로 집계되는 것 방지, 7종 → 6종)
3. item '케익' → '케이크' → 통일 (상품별 KPI가 둘로 쪼개지는 것 방지, 6종 → 5종)
4. revenue 음수 → 별도 보관 1건 (refunds_bakery에 보관하고 본 분석에서 제외)
5. revenue 결측 58건 → 제거 (이유: 결측률이 3% 미만으로 낮고, 값을 채우면 존재하지 않는 매출이 생겨 KPI가 왜곡됨)

## 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: S06, 2025-04 (9,156,000원)
- (상품 관점) 매출 1위 상품: 쿠키 (26,690,000원, 365건). 2위 샌드위치(26,297,000원)와의 차이는 약 39만 원

## 5. 한계와 후속 작업
- 모범 답안의 환불 필터(revenue >= 0)는 NaN도 False로 판정해 결측 행까지 함께 지우는 문제가 있어, 본문 예제처럼 `| isna()`를 붙여 환불 분리와 결측 제거의 책임을 나눔
- 고유 키(주문 ID, 시간대)가 없어 중복 321건 중 실제 중복과 우연히 겹친 정상 거래를 구분하지 못함
- quantity=50은 그대로 두었으므로, 대량 주문인지 입력 오류인지 확인이 필요함 (KPI가 부풀려졌을 수 있음)
- 월별 매출이 사실상 하루치 데이터라 월별 추세를 해석하는 데 한계가 있음
- 환불이 1건뿐이라 원인 분석은 데이터를 더 모은 뒤 가능함